# Week 11

break things down into smaller tasks and goals.

In [21]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# load imports
import numpy as np
import pandas as pd
import pickle
import re
from pathlib import Path

from itertools import product

from scipy.sparse import csr_matrix, hstack, diags
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

from surprise import SVD
import ollama

# load user-item interactions
user_item_matrix = pd.read_csv('csv_files/user_item_matrix.csv', index_col=0)
# load train and test
train_df = pd.read_csv('csv_files/preprocessed_train_video_games.csv')
test_df = pd.read_csv('csv_files/preprocessed_test_video_games.csv')

### 1) Collaborative Filtering Base
1. Use your strongest CF setup from previous weeks (e.g., KNN or SVD). 
1b. use the saved SVD model that was pickled
2. Generate predicted scores for unrated (user, item) pairs.
3. Keep full candidate ranking or at least top-K candidates per user.

In [23]:
# load best svd model (user-item matrix with predictions)
u_i_matrix_svd = pd.read_csv("csv_files/user_item_matrix_svd.csv", index_col=0)
u_i_matrix_svd.info()
# check for null cells
u_i_matrix_svd.isnull().sum().sum()

# load unobserved predictions svd
unobserved_preds_svd = pd.read_csv("csv_files/unobserved_predictions_svd.csv", index_col=0)
# sort by rating in descending order
unobserved_preds_svd = unobserved_preds_svd.sort_values("svd_rating", ascending=False)
unobserved_preds_svd.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1389 entries, AE25ZDXYBK3LHKCZ7XUODANPME4A to AHZYXDJ3HNLKS2E73VOSNIZZJT4Q
Columns: 932 entries, B00000JRSB to B0C5K4M7WJ
dtypes: float64(932)
memory usage: 9.9+ MB


,item_id,svd_rating
user_id,,
AELD7NXSVVFKTNFB3I673NGJVSWA,B00J4Y6L2Y,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B017W1771Y,5.0
AHMG3HUA4K6H475VQGJF2QAOTFTQ,B0BDWVBWC9,5.0
AF52HVGBBCP3JAS6B7B3I4AVIWRQ,B004QEV0MI,5.0
AGP7FLSJ7BHYFVZY3FRKVVDBH47Q,B007CSF3GO,5.0


#### I will keep all 932 entries for full analysis or not idk
### DO a SUBSET for easier time matching common item between all the sets and lower compuational cost
## dumbahh

### 2) Content-Based Base (Original Metadata)
1. Implement same CB representation as Week 10.
1b. rerun week 10 and save the content based model as pickle file
2. prompt an agent and ask how I can create an unobserved df frame from the predicitions in week 10

### 3) Content-Based Base (LLM Metadata)
1. For each item title, generate a short, structured description using Gemma via Ollama.
2. Create text features from generated descriptions (same vectorization pipeline used in CB A where possible).
3. Compute CB scores for unrated items.


#### so many trade offs and decisions
how long should the description be?
what should it included?
how detailed? ect.

In [ ]:
metadata_df = pd.read_parquet('datasets/meta_video_games.parquet')
metadata_df.head()

In [30]:
# filter by item ids in test set
metadata_df = metadata_df[metadata_df['item_id'].isin(test_df['item_id'])]
metadata_df.head()
print(len(metadata_df), len(test_df["item_id"].unique()))

927 927


In [ ]:
# 3) Content-Based Base (LLM Metadata) - Gemma description generation

metadata_llm_df = metadata_df.copy()
metadata_llm_df["title"] = metadata_llm_df["title"].fillna("").astype(str).str.strip()
metadata_llm_df = metadata_llm_df[metadata_llm_df["title"] != ""].copy()
items_df = metadata_llm_df[["item_id", "title"]].drop_duplicates(subset=["item_id"]).reset_index(drop=True)

cache_path = Path("csv_files/llm_item_descriptions.csv")
model_name = "gemma3"
deterministic_options = {
    "temperature": 0.0,
    "top_p": 1.0,
    "seed": 42,
}

def generate_short_description(title_text: str) -> str:
    prompt = (
        "Write 1-2 concise sentences describing this video game title for recommender metadata. "
        "Focus on likely genre, gameplay style, and intended audience. "
        "Do not use bullet points. Title: " + title_text
    )
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "You generate concise, neutral recommendation metadata."},
            {"role": "user", "content": prompt},
        ],
        options=deterministic_options,
    )
    return response["message"]["content"].strip()

if cache_path.exists():
    cached_df = pd.read_csv(cache_path, dtype={"item_id": str})
    cached_df = cached_df[["item_id", "title", "llm_description"]].drop_duplicates(subset=["item_id"])
else:
    cached_df = pd.DataFrame(columns=["item_id", "title", "llm_description"])

items_df["item_id"] = items_df["item_id"].astype(str)
done_item_ids = set(cached_df["item_id"].astype(str))
todo_df = items_df[~items_df["item_id"].isin(done_item_ids)].copy()

print(f"Total items: {len(items_df)}")
print(f"Already cached: {len(cached_df)}")
print(f"To generate now: {len(todo_df)}")

new_rows = []
for i, row in enumerate(todo_df.itertuples(index=False), start=1):
    item_id = str(row.item_id)
    title_text = row.title
    try:
        desc = generate_short_description(title_text)
    except Exception as exc:
        desc = ""
        print(f"[WARN] generation failed for item_id={item_id}: {exc}")

    new_rows.append({
        "item_id": item_id,
        "title": title_text,
        "llm_description": desc,
    })

    if i % 25 == 0:
        print(f"Generated {i}/{len(todo_df)} new descriptions...")

if new_rows:
    new_df = pd.DataFrame(new_rows)
    final_df = pd.concat([cached_df, new_df], ignore_index=True)
else:
    final_df = cached_df.copy()

final_df = final_df.drop_duplicates(subset=["item_id"], keep="last")
final_df = final_df.sort_values("item_id").reset_index(drop=True)
final_df.to_csv(cache_path, index=False)

llm_descriptions_df = final_df.copy()
print(f"Saved cache to: {cache_path}")
print(f"Cached descriptions: {len(llm_descriptions_df)}")
llm_descriptions_df.head()

#### bro never thinks about computational costs for calling a local LLM 
you don't think about the implication of that many rows
just do a top k bruh